In [ ]:
#%matplotlib ipympl

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgba
import colormaps as cmaps

from multiprocessing import Pool
from tqdm import tqdm

import xarray as xr
import numpy as np

In [ ]:
def plot_alpha_lines(line_x, line_y, colors, alpha, linewidth=1, ax=None):
    """
    Plot lines with different colors and alpha values,
    to make the tail "disappear" (magic!).
    Uses `LineCollection` for fast plotting.
    
    Parameters:
    ----------
    line_x : np.ndarray
        x-coordinates, dimensions `(n_lines, n_segments)`.
    line_y : np.ndarray
        y-coordinates, dimensions `(n_lines, n_segments)`.
    colors : str, list or np.ndarray
        Colors of each line, options:
        - Single color string: 'k', 'r', etc. (applied to all lines).
        - List of color strings: ['r', 'b', 'g'], dimensions `(n_lines,)`.
        - Array of RGBA values: shape `(n_lines, 4)`.
    alpha : np.ndarray
        Alpha of each line segment, dimensions `(n_segments,)`.
    linewidth : float, optional
        Line width for all lines. Default is 2.
    ax : matplotlib axis, optional
        Axis to plot on. If None, uses current axis.
    
    Returns:
    -------
    None
    """
    n_lines, n_segments = line_x.shape
    
    # Convert colors to RGBA array (n_lines, 4).
    if isinstance(colors, str):
        # Single color for all lines
        line_colors = np.array([to_rgba(colors)] * n_lines)
    elif isinstance(colors, (list, tuple)):
        line_colors = np.array([to_rgba(c) for c in colors])
    elif isinstance(colors, np.ndarray):
        if colors.ndim == 1:
            line_colors = np.array([to_rgba(c) for c in colors])
        elif colors.shape[1] == 4:
            line_colors = colors
    
    # Stack x and y into points, then create segments.
    # points: (n_lines, n_segments, 2)
    points = np.stack([line_x, line_y], axis=2)

    # Segments: (n_lines, n_segments-1, 2, 2) where each segment is [[x0,y0], [x1,y1]].
    segments = np.stack([points[:, :-1], points[:, 1:]], axis=2)
    
    # Broadcast to create RGBA array: shape (n_lines, n_segments-1, 4).
    n_seg_actual = n_segments - 1
    rgba_colors = np.zeros((n_lines, n_seg_actual, 4))
    rgba_colors[:, :, :3] = line_colors[:, np.newaxis, :3]
    rgba_colors[:, :, 3] = alpha[:n_seg_actual][np.newaxis, :]
    
    # Flatten for LineCollection.
    segments_flat = segments.reshape(-1, 2, 2)
    colors_flat = rgba_colors.reshape(-1, 4)
    
    # Create line collection for fast plotting.
    lc = LineCollection(segments_flat, colors=colors_flat, linewidths=linewidth)
    
    # Plot!
    if ax is None:
        ax = plt.gca()
    ax.add_collection(lc)
    ax.autoscale()

In [ ]:
"""
Read particle dump.
"""

ds = xr.open_dataset('particle_dump.0000000.h5', engine='h5netcdf')
ds = ds.sel(time=slice(1800, 7200))

In [ ]:
"""
Find particles that don't cross the cyclic BCs in the x-direction.
"""

x = ds.x.values
y = ds.y.values
z = ds.z.values

dx = np.diff(x, axis=0)
crosses_boundary = np.any(np.abs(dx) > 1000, axis=0)
x = x[:, ~crosses_boundary]
y = y[:, ~crosses_boundary]
z = z[:, ~crosses_boundary]

time = ds.time.values

In [ ]:
"""
Plot animation frames.
"""

def plot_frame(t):
    fig=plt.figure(figsize=(19.2, 10.8))
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax=plt.gca()
    ax.set_axis_off()

    t1 = t+2
    t0 = max(0, t1-n_time)

    alpha_local = np.linspace(0, 1, t1-t0)**2

    plot_alpha_lines(x[t0:t1].T, z[t0:t1].T, colors, alpha_local, line_width)

    plt.xlim(0, 2500)
    plt.ylim(0, 1000)

    plt.text(20, 970, f't = {np.round(time[t]):.0f} sec.', size=12)
    plt.text(20, 20, 'Simulated with MicroHH (microhh.org)', size=12)

    plt.savefig(f'figs/fig{t:05d}.png')
    plt.close(fig)
    return t

plt.close('all')
plt.ioff()

cmap = cmaps.WhiteBlueGreenYellowRed
line_width = 2.5

n_time = 75
n_part = x.shape[1]

colors = cmap(np.linspace(0.1, 1, n_part))

# Parallel processing.
with Pool() as pool:
    for result in tqdm(pool.imap(plot_frame, range(x.shape[0])), total=x.shape[0]):
        pass

In [ ]:
"""
Create MP4 with FFMPEG.
"""

!ffmpeg -y -framerate 20 -i figs/fig%05d.png -s:v 1920x1080 -c:v libx264 -profile:v high -crf 15 -pix_fmt yuv420p -r 25 barticles.mp4